# Build the Claude classification queue

Prep for Claude classifications.

Reads `classification_input_combined_*.csv`, dedupes to one row per
unique classification unit, and writes a resumable queue that the Claude batch step works
through. Not-employed individuals are auto-resolved here with no search and no LLM call, mirroring `0902`'s
`NOT_EMPLOYED_RESULT` shortcut.

Unit key logic:
- Informative employer present -> `(name, employer)` — same as the existing `0901` `make_key()`.
- Employer blank/uninformative but occupation present (self-employed/sole-proprietor) -> `(name, occupation)` —
  disambiguates two self-employed people with the same name in different fields, which the existing pipeline's
  key collapses into one unit.
- Not-employed individuals (employer AND occupation both blank or in the `NOT_EMPLOYED` set) -> auto-resolved,
  no key needed for searching.

In [8]:
import sys
sys.path.insert(0, ".")

import pandas as pd

from standardization_helpers import (
    normalize_for_key,
    normalize_employer_for_key,
    NOT_EMPLOYED,
    lightly_process_employer,
    standardize_occupation_employer,
)

In [9]:
INPUT_PATH = "../01_contributor_pipeline/01_outputs/classification_input_combined_09-08-26.csv"
QUEUE_PATH = "03_outputs/claude_classification_queue.csv"

In [10]:
df = pd.read_csv(INPUT_PATH, dtype=str, low_memory=False).fillna("")
print(f"Loaded {len(df):,} raw rows from {INPUT_PATH}")
df[["Contributor.Name", "Contributor.Employer", "standardized_employer_name", "processed_occupation", "entity_type", "Amount"]].head()

Loaded 42,491 raw rows from ../01_contributor_pipeline/01_outputs/classification_input_combined_09-08-26.csv


,Contributor.Name,Contributor.Employer,standardized_employer_name,processed_occupation,entity_type,Amount
0,Pursue Health LLC(Zachary Larson),,,,organization,49.5
1,International Longshore and Warehouse Union Fo...,,,,organization,300
2,"DENISON, DARLENE",STATE FARM,STATE FARM,INSURANCE AGENT,individual,300
3,Cypress Healthcare Group,,,,organization,49.33
4,"RNLAWAD, LLC AND AFFILIATED ENTITIES(RESPONSIB...",,,,organization,631.99


In [11]:
def _is_not_employed(employer_proc: str, occ_proc: str) -> bool:
    e = (employer_proc or "").strip().lower()
    o = (occ_proc or "").strip().lower()
    return (not e or e in NOT_EMPLOYED) and (not o or o in NOT_EMPLOYED)


def build_unit(row: dict) -> dict:
    """
    Returns the search_key plus a skip_search flag and (for skip_search=True rows)
    the pre-filled classification. Extends the existing 0901 `make_key()` logic
    with occupation-based disambiguation for the self-employed/blank-employer case.
    """
    entity_type = (row.get("entity_type", "") or "").strip()
    name_key = normalize_for_key(row.get("Contributor.Name", "") or "")
    std_emp = (row.get("standardized_employer_name", "") or "").strip()
    raw_emp = (row.get("Contributor.Employer", "") or "").strip()
    occ_proc = (row.get("processed_occupation", "") or "").strip()

    if not name_key:
        # No name to search on at all -- extremely rare, but can't be searched.
        return {
            "search_key": "BLANK_NAME",
            "skip_search": True,
            "claude_entity_type": "",
            "naics_code_claude": "99",
            "naics_confidence": "unknown",
            "industry_summary": "No identifying information available",
            "reasoning": "Contributor name is blank; nothing to search on.",
        }

    emp_proc = (
        lightly_process_employer(std_emp)
        if (std_emp or entity_type == "organization")
        else standardize_occupation_employer(raw_emp.upper())
    )
    employer_key = normalize_employer_for_key(raw_emp)

    if entity_type == "individual" and _is_not_employed(emp_proc, occ_proc):
        return {
            "search_key": f"{name_key}|NOT_EMPLOYED",
            "skip_search": True,
            "claude_entity_type": "individual",
            "naics_code_claude": "100",
            "naics_confidence": "high",
            "industry_summary": "Not employed",
            "reasoning": "Employer and occupation both indicate not employed/retired/student/etc.; code fixed at 100.",
        }

    if entity_type == "individual" and not employer_key and occ_proc:
        # Self-employed / sole proprietor with no employer -- key on occupation
        # instead, so different occupations under a blank employer don't collide.
        occ_key = normalize_for_key(occ_proc)
        return {"search_key": f"{name_key}||{occ_key}", "skip_search": False}

    # Default: informative employer (individuals) or org name (organizations)
    return {"search_key": f"{name_key}|{employer_key}", "skip_search": False}

In [17]:
units = df.apply(lambda r: build_unit(r.to_dict()), axis=1, result_type="expand")
df = pd.concat([df, units], axis=1)
print(f"{df['search_key'].nunique():,} unique search keys across {len(df):,} raw rows")
print(f"  of which skip_search=True (auto-resolved, no search needed): {df.drop_duplicates('search_key')['skip_search'].sum():,}")

TypeError: unsupported format string passed to Series.__format__

In [13]:
agg = (
    df.groupby("search_key")
    .agg(
        standardized_name=("standardized_name", "first"),
        standardized_employer_name=("standardized_employer_name", "first"),
        standardized_occupation=("processed_occupation", "first"),
        standardized_city=("standardized_city", "first"),
        Contributor_State=("Contributor.State", "first"),
        entity_type=("entity_type", "first"),
        n_raw_rows=("search_key", "size"),
        total_amount=("Amount", lambda s: pd.to_numeric(s, errors="coerce").sum()),
        skip_search=("skip_search", "first"),
        claude_entity_type=("claude_entity_type", "first"),
        naics_code_claude=("naics_code_claude", "first"),
        naics_confidence=("naics_confidence", "first"),
        industry_summary=("industry_summary", "first"),
        reasoning=("reasoning", "first"),
    )
    .reset_index()
)

agg["entity_type_mismatch"] = ""
agg["urls"] = ""
agg["batch_id"] = ""
agg["processed_at"] = ""
agg["status"] = agg["skip_search"].map(lambda s: "done" if s else "pending")

print(f"{len(agg):,} unique units in the queue")
agg["status"].value_counts()

5,294 unique units in the queue


status
pending    4686
done        608
Name: count, dtype: int64

In [14]:
# Sanity checks before saving
print("By entity_type (existing rule-based label):")
print(agg["entity_type"].value_counts(dropna=False))
print()
print("By skip_search:")
print(agg["skip_search"].value_counts(dropna=False))
print()
print("Sample of units needing search (status=pending):")
agg[agg["status"] == "pending"][["search_key", "standardized_name", "standardized_employer_name", "standardized_occupation", "entity_type", "n_raw_rows", "total_amount"]].sample(10, random_state=42)

By entity_type (existing rule-based label):
entity_type
organization    2666
individual      2628
Name: count, dtype: int64

By skip_search:
skip_search
False    4686
True      608
Name: count, dtype: int64

Sample of units needing search (status=pending):


,search_key,standardized_name,standardized_employer_name,standardized_occupation,entity_type,n_raw_rows,total_amount
4297,SHIRALIAN ENTERPRISES INC|,SHIRALIAN ENTERPRISES INC,,,organization,3,138000.00
5286,ZILLOW INC|,ZILLOW INC,,,organization,6,47000.00
884,CHEN BRIAN|ZIP INC,"CHEN, BRIAN","ZIP, INC",SOFTWARE ENGINEER,individual,1,39200.00
4523,STOPPELMAN MICHAEL|SUSA VENTURES,"STOPPELMAN, MICHAEL",SUSA VENTURES,ADVISOR,individual,2,10000.00
3978,RUBIN NANCY|REGINA CHINWEZE ACCOUNTANT,"RUBIN, NANCY","REGINA CHINWEZE, ACCOUNTANT",ACCOUNTANT,individual,2,100.00
4114,SANTE HEALTH SYSTEM INC|,"SANTE HEALTH SYSTEM, INC",,,organization,14,8499.90
2717,LOFGREN PETER|ANTHROPIC,"LOFGREN, PETER",ANTHROPIC,MEMBER OF TECHNICAL STAFF,individual,1,9800.00
549,BOYCE SARAH|AVIDITY BIOLOGY,"BOYCE, SARAH",AVIDITY BIOLOGY,CEO,individual,2,7030.26
2264,JACKSON GREG|THE SIMON LAW GROUP,"JACKSON, GREG",THE SIMON LAW GROUP,ATTORNEY,individual,1,5000.00
4987,VALOV BROTHERS FARMS LP|,"VALOV BROTHERS FARMS, LP",,,organization,2,11000.00


In [15]:
agg = agg.rename(columns={"Contributor_State": "Contributor.State"})
cols = [
    "search_key", "standardized_name", "standardized_employer_name", "standardized_occupation",
    "standardized_city", "Contributor.State", "entity_type", "n_raw_rows", "total_amount",
    "skip_search", "status", "claude_entity_type", "entity_type_mismatch", "industry_summary",
    "naics_code_claude", "naics_confidence", "urls", "reasoning", "batch_id", "processed_at",
]
agg = agg[cols]
agg.to_csv(QUEUE_PATH, index=False)
print(f"Wrote {len(agg):,} rows to {QUEUE_PATH}")

Wrote 5,294 rows to 03_outputs/claude_classification_queue.csv
